In [8]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.3.5"
!pip install pycaret==3.3.2

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-3.3.5.tar.gz (1.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl (28.9 MB)
Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl (9.5 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for lightgbm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [104 lines of output]
      /private/var/folders/y0/9zbck24x5017gh20l_rnx3wr0000gn/T/pip-build-env-4i0vide1/overlay/lib/python3.9/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are d

In [21]:
import pandas as pd
from pycaret.regression import *
from sklearn.model_selection import train_test_split

In [38]:
SEED = 128
target = "england_wales_demand"

df = pd.read_csv("../Dataset2_Demand/6_Elec_Demand_Final.csv")

In [39]:
df["settlement_date"] = pd.to_datetime(df["settlement_date"])
df["month"] = df["settlement_date"].dt.month
df["day_of_week"] = df["settlement_date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

# Capacity utilization ratios
df["wind_utilization"] = df["embedded_wind_generation"] / (df["embedded_wind_capacity"] + 1)
df["solar_utilization"] = df["embedded_solar_generation"] / (df["embedded_solar_capacity"] + 1)

# Net cross-border flow sum
flow_cols = ["ifa2_flow","britned_flow","moyle_flow","east_west_flow","nemo_flow"]
df["net_crossborder_flow"] = df[flow_cols].sum(axis=1)

In [40]:
df = df.sample(50000, random_state=SEED)  # 50k rijen

In [35]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=SEED,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1/3,
    random_state=SEED,
    shuffle=True
)

In [41]:
reg = setup(
    data=df,
    target=target,
    session_id=SEED,
    normalize=True,
    polynomial_features=False,
    transform_target=True,
    remove_multicollinearity=True,
    multicollinearity_threshold=0.9,
    use_gpu=False,
    html=False
)

                    Description                 Value
0                    Session id                   128
1                        Target  england_wales_demand
2                   Target type            Regression
3           Original data shape           (50000, 23)
4        Transformed data shape           (50000, 23)
5   Transformed train set shape           (35000, 23)
6    Transformed test set shape           (15000, 23)
7              Numeric features                    21
8                 Date features                     1
9                    Preprocess                  True
10              Imputation type                simple
11           Numeric imputation                  mean
12       Categorical imputation                  mode
13     Remove multicollinearity                  True
14  Multicollinearity threshold                   0.9
15                    Normalize                  True
16             Normalize method                zscore
17             Transform tar

In [42]:
best_model = compare_models(sort="MAE", n_select=1, turbo=True)


Processing:   0%|          | 0/77 [00:00<?, ?it/s]

                                    Model        MAE           MSE       RMSE  \
lightgbm  Light Gradient Boosting Machine   132.3822  7.088892e+04   230.1669   
et                  Extra Trees Regressor   136.5011  7.138588e+04   230.7402   
rf                Random Forest Regressor   140.8978  8.392067e+04   258.6523   
gbr           Gradient Boosting Regressor   152.6868  7.927263e+04   248.9086   
dt                Decision Tree Regressor   200.5067  2.141233e+05   419.6629   
huber                     Huber Regressor   239.2182  1.327369e+05   346.6215   
ridge                    Ridge Regression   241.2694  1.316798e+05   345.0970   
br                         Bayesian Ridge   241.2795  1.316955e+05   345.1210   
lar                Least Angle Regression   241.2801  1.316965e+05   345.1226   
lr                      Linear Regression   241.2999  1.316699e+05   345.0863   
lasso                    Lasso Regression   254.4982  1.405630e+05   358.4478   
llar         Lasso Least Ang

In [43]:
tuned_model = tune_model(
    best_model, 
    optimize="MAE", 
    fold=5,
    n_iter=20
)


Fitting 5 folds for each of 20 candidates, totalling 100 fits


           MAE          MSE      RMSE      R2   RMSLE    MAPE
Fold                                                         
0     131.8808   36703.4193  191.5814  0.9993  0.0060  0.0042
1     128.3793   31615.0295  177.8062  0.9994  0.0058  0.0041
2     129.7648   34281.7093  185.1532  0.9993  0.0060  0.0041
3     134.0015  215011.7607  463.6936  0.9958  0.1255  0.0041
4     129.5395   34871.2854  186.7385  0.9993  0.0060  0.0041
Mean  130.7132   70496.6408  240.9946  0.9986  0.0299  0.0041
Std     1.9950   72275.9553  111.4372  0.0014  0.0478  0.0000


In [48]:
final_model = finalize_model(tuned_model)

In [49]:
predictions = predict_model(final_model)
print(predictions.head())

[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
                             Model       MAE         MSE      RMSE      R2  \
0  Light Gradient Boosting Machine  156.0794  90990.0131  301.6455  0.9983   

    RMSLE    MAPE  
0  0.0133  0.0049  
       settlement_date  settlement_period     nd           tsd  \
57002       2004-04-02                 29  43359  35683.097656   
206749      2012-10-17                 23  40393  41158.000000   
274081      2016-08-20                 11  18247  20519.000000   
182089      2011-05-22                 35  32263  33185.000000   
258604      2015-10-02                 38  38114  39184.000000   

        embedded_wind_generation  embedded_wind_capacity 

In [50]:
save_model(final_model, "england_wales_demand_predictor")

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('target_transformation',
                  TransformerWrapperWithInverse(transformer=TargetTransformer(estimator=PowerTransformer(standardize=False)))),
                 ('date_feature_extractor',
                  TransformerWrapper(include=['settlement_date'],
                                     transformer=ExtractDateTimeFeatures())),
                 ('numerical_imputer',
                  TransformerWrapper(include=['settleme...
                                     transformer=RemoveMulticollinearity(threshold=0.9))),
                 ('normalize', TransformerWrapper(transformer=StandardScaler())),
                 ('actual_estimator',
                  LGBMRegressor(bagging_fraction=1.0, bagging_freq=3,
                                feature_fraction=0.6, learning_rate=0.2,
                                min_child_samples=1, min_split_gain=0.4,
                                n_estimators=290, n_jobs=-1, num_leaves=70,
